In [ ]:
# ============================================================
# AIMES: MODEL-SPECIFIC MFT VALUE DIRECTION EXTRACTION
#
# Script 1
#
# Supported models:
#
#   gemma-4b   -> google/gemma-3-4b-it
#   qwen-4b    -> Qwen/Qwen3-4B
#   llama-8b   -> meta-llama/Llama-3.1-8B-Instruct
#   gemma-12b  -> google/gemma-3-12b-it
#   qwen-14b   -> Qwen/Qwen3-14B
#
# Computes for each foundation k and layer l:
#
#   mu^+_{k,l}
#   mu^-_{k,l}
#   raw_direction = mu^+ - mu^-
#   unit_direction = normalize(raw_direction)
#
# Representation:
#   final non-padding input-token residual activation
#   from every transformer layer.
#
# Saves:
#
#   activations/
#       per-foundation positive/negative activation caches
#
#   directions/
#       model-specific direction safetensors
#       model-specific raw direction norm CSV
#
#   diagnostics/
#       raw_direction_norms.csv
#
#   metadata/
#       dataset copy
#       metadata JSON
#
#   logs/
#       extraction log
#
# Upload destination:
#
#   <HF_USERNAME>/AIMES
#
# ============================================================


# ============================================================
# 0. INSTALL
# ============================================================

!pip install -q -U \
    transformers \
    accelerate \
    huggingface_hub \
    safetensors \
    "pandas==2.2.3"


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import shutil
import logging
from datetime import datetime, timezone

import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
)

from huggingface_hub import (
    HfApi,
    create_repo,
)

from safetensors.torch import save_file

from google.colab import userdata
from google.colab import drive


drive.mount(
    "/content/drive",
    force_remount=True
)


# ============================================================
# 2. MANUALLY SELECT MODEL
# ============================================================

# Change ONLY this line for each run.

MODEL_KEY = "gemma-4b"
MODEL_KEY = "qwen-4b"
MODEL_KEY = "llama-8b"
MODEL_KEY = "gemma-12b"
MODEL_KEY = "qwen-14b"


MODEL_CONFIGS = {

    "gemma-4b": {
        "model_id": "google/gemma-3-4b-it",
        "save_name": "gemma-3-4b-it",
        "family": "gemma3",
        "batch_size": 4,
    },

    "qwen-4b": {
        "model_id": "Qwen/Qwen3-4B",
        "save_name": "qwen3-4b",
        "family": "qwen3",
        "batch_size": 4,
    },

    "llama-8b": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "save_name": "llama-3.1-8b-instruct",
        "family": "llama",
        "batch_size": 4,
    },

    "gemma-12b": {
        "model_id": "google/gemma-3-12b-it",
        "save_name": "gemma-3-12b-it",
        "family": "gemma3",
        "batch_size": 2,
    },

    "qwen-14b": {
        "model_id": "Qwen/Qwen3-14B",
        "save_name": "qwen3-14b",
        "family": "qwen3",
        "batch_size": 2,
    },
}


assert MODEL_KEY in MODEL_CONFIGS, (
    f"Unknown MODEL_KEY: {MODEL_KEY}"
)


MODEL_INFO = MODEL_CONFIGS[
    MODEL_KEY
]


MODEL_ID = MODEL_INFO[
    "model_id"
]


MODEL_SAVE_NAME = MODEL_INFO[
    "save_name"
]


MODEL_FAMILY = MODEL_INFO[
    "family"
]


BATCH_SIZE = MODEL_INFO[
    "batch_size"
]


print("=" * 80)
print("SELECTED MODEL")
print("=" * 80)

print("MODEL_KEY  :", MODEL_KEY)
print("MODEL_ID   :", MODEL_ID)
print("SAVE_NAME  :", MODEL_SAVE_NAME)
print("FAMILY     :", MODEL_FAMILY)
print("BATCH_SIZE :", BATCH_SIZE)


# ============================================================
# 3. EXPERIMENT SETTINGS
# ============================================================

FOUNDATIONS = [
    "Care",
    "Fairness",
    "Loyalty",
    "Authority",
    "Sanctity",
]


N_PAIRS_PER_FOUNDATION = 200

MAX_LENGTH = 128

VERSION = "v1"


# Save H+ and H- because Script 2 uses them for:
#
# - held-out projection separation
# - split-half stability
# - inter-value geometry
# - additional direction diagnostics

SAVE_SAMPLE_ACTIVATIONS = True


ACTIVATION_SAVE_DTYPE = (
    torch.float16
)


# ============================================================
# 4. INPUT DATASET
# ============================================================

DATA_PATH = (
    "/content/drive/MyDrive/AIMES/"
    "data/contrastive/"
    "mft_positive_negative_v1.csv"
)


assert os.path.exists(
    DATA_PATH
), (
    f"Dataset not found:\n{DATA_PATH}"
)


# ============================================================
# 5. HUGGING FACE LOGIN
# ============================================================

HF_TOKEN = userdata.get(
    "HF_TOKEN"
)


assert HF_TOKEN is not None, (
    "HF_TOKEN not found in Colab Secrets."
)


api = HfApi(
    token=HF_TOKEN
)


who = api.whoami()


HF_USERNAME = who[
    "name"
]


HF_REPO_ID = (
    f"{HF_USERNAME}/AIMES"
)


REPO_PRIVATE = True


create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=REPO_PRIVATE,
    exist_ok=True,
    token=HF_TOKEN,
)


print(
    "\nHF repository:",
    HF_REPO_ID
)


# ============================================================
# 6. HF DIRECTORY STRUCTURE
# ============================================================

HF_MODEL_ROOT = (
    f"artifacts/value_directions/"
    f"{MODEL_SAVE_NAME}/{VERSION}"
)


HF_ACTIVATION_ROOT = (
    f"{HF_MODEL_ROOT}/activations"
)


HF_DIRECTION_ROOT = (
    f"{HF_MODEL_ROOT}/directions"
)


HF_DIAGNOSTIC_ROOT = (
    f"{HF_MODEL_ROOT}/diagnostics"
)


HF_METADATA_ROOT = (
    f"{HF_MODEL_ROOT}/metadata"
)


HF_LOG_ROOT = (
    f"{HF_MODEL_ROOT}/logs"
)


HF_DATA_ROOT = (
    "data/contrastive"
)


# ============================================================
# 7. LOCAL TEMP DIRECTORY
# ============================================================

LOCAL_ROOT = (
    f"/content/"
    f"aimes_{MODEL_SAVE_NAME}_{VERSION}"
)


if os.path.exists(
    LOCAL_ROOT
):

    shutil.rmtree(
        LOCAL_ROOT
    )


os.makedirs(
    LOCAL_ROOT,
    exist_ok=True
)


LOCAL_CACHE = os.path.join(
    LOCAL_ROOT,
    "hf_cache"
)


os.makedirs(
    LOCAL_CACHE,
    exist_ok=True
)


# ============================================================
# 8. LOGGING
# ============================================================

LOG_PATH = os.path.join(
    LOCAL_ROOT,
    (
        f"value_direction_"
        f"{MODEL_SAVE_NAME}_"
        f"{VERSION}.log"
    ),
)


logger = logging.getLogger(
    f"AIMES_{MODEL_SAVE_NAME}"
)


logger.setLevel(
    logging.INFO
)


logger.handlers.clear()


file_handler = logging.FileHandler(
    LOG_PATH
)


stream_handler = logging.StreamHandler()


formatter = logging.Formatter(
    "%(asctime)s | "
    "%(levelname)s | "
    "%(message)s"
)


file_handler.setFormatter(
    formatter
)


stream_handler.setFormatter(
    formatter
)


logger.addHandler(
    file_handler
)


logger.addHandler(
    stream_handler
)


logger.info(
    f"Starting value direction extraction: "
    f"{MODEL_ID}"
)


# ============================================================
# 9. LOAD DATASET
# ============================================================

df = pd.read_csv(
    DATA_PATH
)


required_columns = {
    "foundation",
    "positive_text",
    "negative_text",
}


missing_columns = (
    required_columns
    -
    set(df.columns)
)


assert not missing_columns, (
    f"Missing columns: "
    f"{missing_columns}"
)


print(
    "\nDataset shape:",
    df.shape
)


# ============================================================
# 10. VERIFY EXACT DUPLICATES
# ============================================================

temp = df.copy()


temp["_pos_norm"] = (
    temp[
        "positive_text"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


temp["_neg_norm"] = (
    temp[
        "negative_text"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


duplicate_mask = (
    temp.duplicated(
        subset=[
            "foundation",
            "_pos_norm",
            "_neg_norm",
        ],
        keep=False
    )
)


n_duplicate_rows = int(
    duplicate_mask.sum()
)


print(
    "Rows involved in exact duplicates:",
    n_duplicate_rows
)


assert (
    n_duplicate_rows
    ==
    0
), (
    "Duplicate contrastive pairs remain."
)


del temp


# ============================================================
# 11. VERIFY FOUNDATION COUNTS
# ============================================================

counts = (
    df[
        "foundation"
    ]
    .value_counts()
    .to_dict()
)


print(
    "\nPairs per foundation:"
)


for foundation in FOUNDATIONS:

    n = counts.get(
        foundation,
        0
    )

    print(
        f"{foundation:12s}: {n}"
    )


for foundation in FOUNDATIONS:

    assert (
        counts.get(
            foundation,
            0
        )
        ==
        N_PAIRS_PER_FOUNDATION
    ), (
        f"{foundation} does not "
        f"contain exactly "
        f"{N_PAIRS_PER_FOUNDATION} pairs."
    )


assert (
    len(df)
    ==
    len(FOUNDATIONS)
    *
    N_PAIRS_PER_FOUNDATION
)


logger.info(
    "Dataset validation passed."
)


# ============================================================
# 12. UPLOAD DATASET TO HF
# ============================================================

dataset_filename = os.path.basename(
    DATA_PATH
)


logger.info(
    "Uploading contrastive dataset..."
)


api.upload_file(

    path_or_fileobj=
        DATA_PATH,

    path_in_repo=(
        f"{HF_DATA_ROOT}/"
        f"{dataset_filename}"
    ),

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


logger.info(
    "Dataset uploaded."
)


# ============================================================
# 13. COMPUTE DTYPE
# ============================================================

if torch.cuda.is_available():

    if torch.cuda.is_bf16_supported():

        MODEL_DTYPE = (
            torch.bfloat16
        )

    else:

        MODEL_DTYPE = (
            torch.float16
        )

else:

    MODEL_DTYPE = (
        torch.float32
    )


print(
    "\nCompute dtype:",
    MODEL_DTYPE
)


# ============================================================
# 14. LOAD MODEL / TOKENIZER
# ============================================================

processor = None


if MODEL_FAMILY == "gemma3":

    logger.info(
        "Loading Gemma 3 processor..."
    )


    processor = (
        AutoProcessor
        .from_pretrained(

            MODEL_ID,

            token=
                HF_TOKEN,

            cache_dir=
                LOCAL_CACHE,
        )
    )


    tokenizer = (
        processor.tokenizer
    )


    logger.info(
        "Loading Gemma 3 model..."
    )


    model = (
        Gemma3ForConditionalGeneration
        .from_pretrained(

            MODEL_ID,

            token=
                HF_TOKEN,

            torch_dtype=
                MODEL_DTYPE,

            device_map=
                "auto",

            cache_dir=
                LOCAL_CACHE,

            low_cpu_mem_usage=
                True,
        )
    )


else:

    logger.info(
        "Loading tokenizer..."
    )


    tokenizer = (
        AutoTokenizer
        .from_pretrained(

            MODEL_ID,

            token=
                HF_TOKEN,

            cache_dir=
                LOCAL_CACHE,
        )
    )


    logger.info(
        "Loading causal LM..."
    )


    model = (
        AutoModelForCausalLM
        .from_pretrained(

            MODEL_ID,

            token=
                HF_TOKEN,

            torch_dtype=
                MODEL_DTYPE,

            device_map=
                "auto",

            cache_dir=
                LOCAL_CACHE,

            low_cpu_mem_usage=
                True,
        )
    )


tokenizer.padding_side = (
    "right"
)


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


model.eval()


logger.info(
    "Model loaded successfully."
)


# ============================================================
# 15. GET TEXT CONFIG
# ============================================================

config = model.config


if hasattr(
    config,
    "text_config"
):

    text_config = (
        config.text_config
    )

else:

    text_config = (
        config
    )


NUM_LAYERS = int(
    text_config.num_hidden_layers
)


HIDDEN_SIZE = int(
    text_config.hidden_size
)


VOCAB_SIZE = int(
    text_config.vocab_size
)


MODEL_REVISION = getattr(
    config,
    "_commit_hash",
    None
)


print(
    "\nModel architecture:"
)


print(
    "Layers      :",
    NUM_LAYERS
)


print(
    "Hidden size :",
    HIDDEN_SIZE
)


print(
    "Vocabulary  :",
    VOCAB_SIZE
)


print(
    "Revision    :",
    MODEL_REVISION
)


logger.info(
    f"layers={NUM_LAYERS}"
)


logger.info(
    f"hidden_size={HIDDEN_SIZE}"
)


logger.info(
    f"vocab_size={VOCAB_SIZE}"
)


logger.info(
    f"revision={MODEL_REVISION}"
)


# ============================================================
# 16. INPUT EMBEDDING DEVICE
# ============================================================

def get_input_device(
    model
):

    try:

        embeddings = (
            model
            .get_input_embeddings()
        )

        return (
            embeddings
            .weight
            .device
        )

    except Exception:

        pass


    if hasattr(
        model,
        "language_model"
    ):

        embeddings = (
            model
            .language_model
            .get_input_embeddings()
        )

        return (
            embeddings
            .weight
            .device
        )


    if hasattr(
        model,
        "model"
    ):

        if hasattr(
            model.model,
            "language_model"
        ):

            embeddings = (
                model
                .model
                .language_model
                .get_input_embeddings()
            )

            return (
                embeddings
                .weight
                .device
            )


    raise RuntimeError(
        "Unable to determine "
        "input embedding device."
    )


INPUT_DEVICE = (
    get_input_device(
        model
    )
)


print(
    "Input device:",
    INPUT_DEVICE
)


# ============================================================
# 17. GET HIDDEN STATES ROBUSTLY
# ============================================================

def get_hidden_states_from_output(
    outputs
):

    # Standard causal LM

    hidden = getattr(
        outputs,
        "hidden_states",
        None
    )

    if hidden is not None:

        return hidden


    # Some wrappers

    language_model_output = getattr(
        outputs,
        "language_model_output",
        None
    )

    if language_model_output is not None:

        hidden = getattr(
            language_model_output,
            "hidden_states",
            None
        )

        if hidden is not None:

            return hidden


    # Gemma-style nested outputs

    text_model_output = getattr(
        outputs,
        "text_model_output",
        None
    )

    if text_model_output is not None:

        hidden = getattr(
            text_model_output,
            "hidden_states",
            None
        )

        if hidden is not None:

            return hidden


    raise RuntimeError(
        "Could not find hidden_states "
        "in model output."
    )


# ============================================================
# 18. MODEL SANITY TEST
# ============================================================

TEST_TEXT = (
    "A student stayed after class "
    "to help a classmate who was upset."
)


test_inputs = tokenizer(

    TEST_TEXT,

    return_tensors=
        "pt",

    add_special_tokens=
        True,
)


test_inputs = {

    k:
        v.to(
            INPUT_DEVICE
        )

    for k, v
    in test_inputs.items()
}


with torch.inference_mode():

    test_outputs = model(

        **test_inputs,

        output_hidden_states=
            True,

        use_cache=
            False,

        return_dict=
            True,
    )


test_hidden_states = (
    get_hidden_states_from_output(
        test_outputs
    )
)


print(
    "\nHidden-state sanity test:"
)


print(
    "Returned hidden states:",
    len(
        test_hidden_states
    )
)


print(
    "Expected:",
    NUM_LAYERS + 1
)


print(
    "Last hidden shape:",
    tuple(
        test_hidden_states[-1].shape
    )
)


assert (
    len(
        test_hidden_states
    )
    ==
    NUM_LAYERS + 1
), (
    "Unexpected number of "
    "hidden-state tensors."
)


assert (
    test_hidden_states[-1]
    .shape[-1]
    ==
    HIDDEN_SIZE
), (
    "Hidden dimension mismatch."
)


logger.info(
    "Hidden-state sanity check passed."
)


del test_outputs
del test_hidden_states
del test_inputs


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 19. ACTIVATION EXTRACTION FUNCTION
# ============================================================

@torch.inference_mode()
def extract_last_token_activations(
    texts
):
    """
    Extract residual hidden states at
    the final non-padding input token.

    Returns
    -------
    Tensor [N, L, d]

    transformers hidden states:
        hidden_states[0]  = embedding output
        hidden_states[1:] = transformer layers 1...L
    """

    all_batches = []


    for start in range(
        0,
        len(texts),
        BATCH_SIZE
    ):

        end = min(
            start
            +
            BATCH_SIZE,

            len(texts)
        )


        batch_texts = (
            texts[
                start:end
            ]
        )


        inputs = tokenizer(

            batch_texts,

            return_tensors=
                "pt",

            padding=
                True,

            truncation=
                True,

            max_length=
                MAX_LENGTH,

            add_special_tokens=
                True,
        )


        attention_mask_cpu = (
            inputs[
                "attention_mask"
            ]
        )


        last_indices = (
            attention_mask_cpu
            .sum(
                dim=1
            )
            -
            1
        )


        inputs = {

            k:
                v.to(
                    INPUT_DEVICE
                )

            for k, v
            in inputs.items()
        }


        outputs = model(

            **inputs,

            output_hidden_states=
                True,

            use_cache=
                False,

            return_dict=
                True,
        )


        all_hidden_states = (
            get_hidden_states_from_output(
                outputs
            )
        )


        # Discard embedding output.

        hidden_states = (
            all_hidden_states[
                1:
            ]
        )


        assert (
            len(
                hidden_states
            )
            ==
            NUM_LAYERS
        )


        batch_size_actual = (
            len(
                batch_texts
            )
        )


        layer_outputs = []


        for layer_hidden in hidden_states:

            idx = (
                last_indices
                .to(
                    layer_hidden.device
                )
            )


            batch_idx = (
                torch.arange(

                    batch_size_actual,

                    device=
                        layer_hidden.device
                )
            )


            last_hidden = (
                layer_hidden[
                    batch_idx,
                    idx,
                    :
                ]
            )


            layer_outputs.append(

                last_hidden
                .detach()
                .float()
                .cpu()
            )


        # Shape [B, L, d]

        batch_activations = (
            torch.stack(
                layer_outputs,
                dim=1
            )
        )


        all_batches.append(
            batch_activations
        )


        logger.info(
            f"Processed "
            f"{end}/{len(texts)} texts"
        )


        del outputs
        del all_hidden_states
        del hidden_states
        del layer_outputs
        del batch_activations
        del inputs


        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    # Shape [N, L, d]

    return torch.cat(
        all_batches,
        dim=0
    )


# ============================================================
# 20. FINAL RESULT STORAGE
# ============================================================

all_mu_positive = []

all_mu_negative = []

all_raw_directions = []

all_unit_directions = []


# ============================================================
# 21. FOUNDATION LOOP
# ============================================================

for foundation in FOUNDATIONS:

    logger.info(
        "=" * 70
    )


    logger.info(
        f"FOUNDATION: {foundation}"
    )


    logger.info(
        "=" * 70
    )


    subset = (

        df[
            df[
                "foundation"
            ]
            ==
            foundation
        ]

        .reset_index(
            drop=True
        )
    )


    assert (
        len(
            subset
        )
        ==
        N_PAIRS_PER_FOUNDATION
    )


    positive_texts = (

        subset[
            "positive_text"
        ]
        .astype(str)
        .tolist()
    )


    negative_texts = (

        subset[
            "negative_text"
        ]
        .astype(str)
        .tolist()
    )


    # ========================================================
    # Positive activation extraction
    # ========================================================

    logger.info(
        "Extracting positive activations..."
    )


    H_pos = (
        extract_last_token_activations(
            positive_texts
        )
    )


    logger.info(
        f"H_pos shape = "
        f"{tuple(H_pos.shape)}"
    )


    # ========================================================
    # Negative activation extraction
    # ========================================================

    logger.info(
        "Extracting negative activations..."
    )


    H_neg = (
        extract_last_token_activations(
            negative_texts
        )
    )


    logger.info(
        f"H_neg shape = "
        f"{tuple(H_neg.shape)}"
    )


    assert H_pos.shape == (

        N_PAIRS_PER_FOUNDATION,
        NUM_LAYERS,
        HIDDEN_SIZE,
    )


    assert H_neg.shape == (

        N_PAIRS_PER_FOUNDATION,
        NUM_LAYERS,
        HIDDEN_SIZE,
    )


    # ========================================================
    # 22. COMPUTE MU+ AND MU-
    # ========================================================

    mu_positive = (
        H_pos.mean(
            dim=0
        )
    )


    mu_negative = (
        H_neg.mean(
            dim=0
        )
    )


    # Shape [L, d]


    # ========================================================
    # 23. RAW DIRECTION
    # ========================================================

    raw_direction = (

        mu_positive
        -
        mu_negative
    )


    # ========================================================
    # 24. UNIT NORMALIZATION
    # ========================================================

    direction_norms = (
        torch.linalg.vector_norm(

            raw_direction,

            ord=2,

            dim=-1,

            keepdim=True
        )
    )


    unit_direction = (

        raw_direction
        /
        direction_norms.clamp_min(
            1e-12
        )
    )


    unit_norm_check = (
        torch.linalg.vector_norm(

            unit_direction,

            dim=-1
        )
    )


    logger.info(
        "Unit direction norm range: "
        f"{unit_norm_check.min().item():.6f}"
        " - "
        f"{unit_norm_check.max().item():.6f}"
    )


    # ========================================================
    # 25. SAVE SAMPLE ACTIVATIONS
    # ========================================================

    if SAVE_SAMPLE_ACTIVATIONS:

        foundation_slug = (
            foundation
            .lower()
            .replace(
                " ",
                "_"
            )
        )


        activation_filename = (
            f"{MODEL_SAVE_NAME}_"
            f"{foundation_slug}_"
            f"activations_{VERSION}."
            f"safetensors"
        )


        activation_path = os.path.join(
            LOCAL_ROOT,
            activation_filename
        )


        activation_payload = {

            "positive":
                H_pos.to(
                    ACTIVATION_SAVE_DTYPE
                ),

            "negative":
                H_neg.to(
                    ACTIVATION_SAVE_DTYPE
                ),
        }


        save_file(
            activation_payload,
            activation_path
        )


        logger.info(
            f"Uploading "
            f"{foundation} activation cache..."
        )


        api.upload_file(

            path_or_fileobj=
                activation_path,

            path_in_repo=(
                f"{HF_ACTIVATION_ROOT}/"
                f"{activation_filename}"
            ),

            repo_id=
                HF_REPO_ID,

            repo_type=
                "model",

            token=
                HF_TOKEN,
        )


        os.remove(
            activation_path
        )


        logger.info(
            f"{foundation} activation "
            f"cache uploaded and "
            f"local copy removed."
        )


    # ========================================================
    # 26. STORE DIRECTION DATA
    # ========================================================

    all_mu_positive.append(
        mu_positive
    )


    all_mu_negative.append(
        mu_negative
    )


    all_raw_directions.append(
        raw_direction
    )


    all_unit_directions.append(
        unit_direction
    )


    # Release sample activations

    del H_pos
    del H_neg

    del mu_positive
    del mu_negative

    del raw_direction
    del unit_direction


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 27. STACK FIVE FOUNDATIONS
# ============================================================

MU_POSITIVE = (
    torch.stack(
        all_mu_positive,
        dim=0
    )
)


MU_NEGATIVE = (
    torch.stack(
        all_mu_negative,
        dim=0
    )
)


RAW_DIRECTIONS = (
    torch.stack(
        all_raw_directions,
        dim=0
    )
)


UNIT_DIRECTIONS = (
    torch.stack(
        all_unit_directions,
        dim=0
    )
)


print(
    "\nFinal tensor shapes:"
)


print(
    "MU_POSITIVE     :",
    tuple(
        MU_POSITIVE.shape
    )
)


print(
    "MU_NEGATIVE     :",
    tuple(
        MU_NEGATIVE.shape
    )
)


print(
    "RAW_DIRECTIONS  :",
    tuple(
        RAW_DIRECTIONS.shape
    )
)


print(
    "UNIT_DIRECTIONS :",
    tuple(
        UNIT_DIRECTIONS.shape
    )
)


assert (
    MU_POSITIVE.shape
    ==
    (
        len(FOUNDATIONS),
        NUM_LAYERS,
        HIDDEN_SIZE,
    )
)


assert (
    UNIT_DIRECTIONS.shape
    ==
    (
        len(FOUNDATIONS),
        NUM_LAYERS,
        HIDDEN_SIZE,
    )
)


# ============================================================
# 28. SAVE FINAL DIRECTION FILE
# ============================================================

direction_filename = (
    f"{MODEL_SAVE_NAME}_"
    f"mft_value_directions_"
    f"{VERSION}.safetensors"
)


DIRECTION_FILE = os.path.join(
    LOCAL_ROOT,
    direction_filename
)


save_file(

    {

        "mu_positive":
            MU_POSITIVE.float(),

        "mu_negative":
            MU_NEGATIVE.float(),

        "raw_directions":
            RAW_DIRECTIONS.float(),

        "unit_directions":
            UNIT_DIRECTIONS.float(),
    },

    DIRECTION_FILE
)


# ============================================================
# 29. RAW DIRECTION-NORM DIAGNOSTIC
# ============================================================

norm_rows = []


for k, foundation in enumerate(
    FOUNDATIONS
):

    norms = (
        torch.linalg.vector_norm(

            RAW_DIRECTIONS[
                k
            ],

            dim=-1
        )
    )


    for layer_idx in range(
        NUM_LAYERS
    ):

        norm_rows.append(

            {
                "model_key":
                    MODEL_KEY,

                "model_id":
                    MODEL_ID,

                "model_save_name":
                    MODEL_SAVE_NAME,

                "foundation":
                    foundation,

                "layer":
                    layer_idx + 1,

                "raw_direction_norm":
                    float(
                        norms[
                            layer_idx
                        ]
                    ),
            }
        )


norm_df = pd.DataFrame(
    norm_rows
)


# ------------------------------------------------------------
# Model-specific filename retained under directions/
# ------------------------------------------------------------

norm_filename = (
    f"{MODEL_SAVE_NAME}_"
    f"direction_norms_"
    f"{VERSION}.csv"
)


NORMS_CSV = os.path.join(
    LOCAL_ROOT,
    norm_filename
)


norm_df.to_csv(
    NORMS_CSV,
    index=False
)


# ------------------------------------------------------------
# Standard downstream diagnostic filename
# ------------------------------------------------------------

RAW_NORMS_DIAGNOSTIC_CSV = os.path.join(
    LOCAL_ROOT,
    "raw_direction_norms.csv"
)


norm_df.to_csv(
    RAW_NORMS_DIAGNOSTIC_CSV,
    index=False
)


print(
    "\nRaw direction norm rows:",
    len(norm_df)
)


# ============================================================
# 30. SAVE DATASET USED FOR THIS MODEL
# ============================================================

used_columns = [

    c

    for c in [

        "pair_id",
        "foundation",
        "context",
        "positive_text",
        "negative_text",

    ]

    if c in df.columns
]


used_dataset_filename = (
    f"{MODEL_SAVE_NAME}_"
    f"direction_dataset_"
    f"{VERSION}.csv"
)


USED_DATA_CSV = os.path.join(
    LOCAL_ROOT,
    used_dataset_filename
)


df[
    used_columns
].to_csv(

    USED_DATA_CSV,

    index=False
)


# ============================================================
# 31. METADATA
# ============================================================

metadata = {

    "project":
        "AIMES",

    "artifact":
        "MFT positive-negative value directions",

    "version":
        VERSION,

    "created_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "model_key":
        MODEL_KEY,

    "model_id":
        MODEL_ID,

    "model_family":
        MODEL_FAMILY,

    "model_revision":
        MODEL_REVISION,

    "model_save_name":
        MODEL_SAVE_NAME,

    "num_hidden_layers":
        NUM_LAYERS,

    "hidden_size":
        HIDDEN_SIZE,

    "vocab_size":
        VOCAB_SIZE,

    "compute_dtype":
        str(
            MODEL_DTYPE
        ),

    "activation_storage_dtype":
        str(
            ACTIVATION_SAVE_DTYPE
        ),

    "foundations":
        FOUNDATIONS,

    "foundation_order":
        FOUNDATIONS,

    "pairs_per_foundation":
        N_PAIRS_PER_FOUNDATION,

    "total_pairs":
        int(
            len(df)
        ),

    "total_vignettes":
        int(
            len(df) * 2
        ),

    "representation":
        (
            "Residual-stream hidden state "
            "at final non-padding input token."
        ),

    "hidden_state_indexing":
        (
            "output_hidden_states[0] is embedding output; "
            "output_hidden_states[1:] correspond "
            "to transformer layers 1...L."
        ),

    "stored_layer_numbering":
        (
            "CSV layer indices use 1...L; "
            "direction tensor index 0 corresponds "
            "to transformer layer 1."
        ),

    "input_format":
        (
            "Raw vignette text tokenized directly; "
            "no chat template used for direction extraction."
        ),

    "max_length":
        MAX_LENGTH,

    "batch_size":
        BATCH_SIZE,

    "direction_definition":
        (
            "normalize("
            "mean(h_positive) - "
            "mean(h_negative)"
            ")"
        ),

    "sample_activations_saved":
        SAVE_SAMPLE_ACTIVATIONS,

    "source_dataset":
        dataset_filename,

    "hf_repo":
        HF_REPO_ID,

    "hf_model_root":
        HF_MODEL_ROOT,
}


metadata_filename = (
    f"{MODEL_SAVE_NAME}_"
    f"metadata_"
    f"{VERSION}.json"
)


METADATA_JSON = os.path.join(
    LOCAL_ROOT,
    metadata_filename
)


with open(
    METADATA_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 32. UPLOAD FINAL DIRECTION ARTIFACTS
# ============================================================

logger.info(
    "Uploading final direction file..."
)


api.upload_file(

    path_or_fileobj=
        DIRECTION_FILE,

    path_in_repo=(
        f"{HF_DIRECTION_ROOT}/"
        f"{direction_filename}"
    ),

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


# ------------------------------------------------------------
# Keep model-specific norm CSV under directions/
# ------------------------------------------------------------

logger.info(
    "Uploading model-specific direction norm CSV..."
)


api.upload_file(

    path_or_fileobj=
        NORMS_CSV,

    path_in_repo=(
        f"{HF_DIRECTION_ROOT}/"
        f"{norm_filename}"
    ),

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


# ------------------------------------------------------------
# Standardized copy under diagnostics/
#
# This is the filename expected by Script 2 / Script 3.
# ------------------------------------------------------------

logger.info(
    "Uploading standardized raw_direction_norms.csv..."
)


api.upload_file(

    path_or_fileobj=
        RAW_NORMS_DIAGNOSTIC_CSV,

    path_in_repo=(
        f"{HF_DIAGNOSTIC_ROOT}/"
        "raw_direction_norms.csv"
    ),

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


# ------------------------------------------------------------
# Dataset metadata copy
# ------------------------------------------------------------

logger.info(
    "Uploading dataset metadata copy..."
)


api.upload_file(

    path_or_fileobj=
        USED_DATA_CSV,

    path_in_repo=(
        f"{HF_METADATA_ROOT}/"
        f"{used_dataset_filename}"
    ),

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


# ------------------------------------------------------------
# Metadata JSON
# ------------------------------------------------------------

logger.info(
    "Uploading metadata JSON..."
)


api.upload_file(

    path_or_fileobj=
        METADATA_JSON,

    path_in_repo=(
        f"{HF_METADATA_ROOT}/"
        f"{metadata_filename}"
    ),

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


# ============================================================
# 33. LOG UPLOAD
# ============================================================

logger.info(
    "Value direction extraction "
    "completed successfully."
)


logger.info(
    f"Artifacts stored at: "
    f"{HF_MODEL_ROOT}"
)


for handler in logger.handlers:

    handler.flush()


log_filename = os.path.basename(
    LOG_PATH
)


api.upload_file(

    path_or_fileobj=
        LOG_PATH,

    path_in_repo=(
        f"{HF_LOG_ROOT}/"
        f"{log_filename}"
    ),

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


# ============================================================
# 34. SUMMARY
# ============================================================

print(
    "\n"
    +
    "=" * 80
)


print(
    "AIMES VALUE DIRECTION EXTRACTION COMPLETE"
)


print(
    "=" * 80
)


print(
    "\nModel:"
)


print(
    MODEL_ID
)


print(
    "\nRevision:"
)


print(
    MODEL_REVISION
)


print(
    "\nLayers:"
)


print(
    NUM_LAYERS
)


print(
    "\nHidden dimension:"
)


print(
    HIDDEN_SIZE
)


print(
    "\nPairs per foundation:"
)


print(
    N_PAIRS_PER_FOUNDATION
)


print(
    "\nDirection tensor shape:"
)


print(
    tuple(
        UNIT_DIRECTIONS.shape
    )
)


print(
    "\nInterpretation:"
)


print(
    "[foundation, layer, hidden_dimension]"
)


print(
    "\nFoundation order:"
)


for i, foundation in enumerate(
    FOUNDATIONS
):

    print(
        i,
        foundation
    )


print(
    "\nHF repository:"
)


print(
    f"https://huggingface.co/"
    f"{HF_REPO_ID}"
)


print(
    "\nArtifact root:"
)


print(
    HF_MODEL_ROOT
)


print(
    "\nDirection file:"
)


print(
    f"{HF_DIRECTION_ROOT}/"
    f"{direction_filename}"
)


print(
    "\nStandardized norm diagnostic:"
)


print(
    f"{HF_DIAGNOSTIC_ROOT}/"
    "raw_direction_norms.csv"
)


print(
    "\nExpected next stage:"
)


print(
    "Run Script 2: model-specific "
    "direction validation."
)


# ============================================================
# 35. CLEAN LOCAL FILES / MODEL CACHE
# ============================================================

del model
del tokenizer


if processor is not None:

    del processor


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# Model weights can occupy many GB in /content.
# Remove them after successful upload.

if os.path.exists(
    LOCAL_CACHE
):

    shutil.rmtree(
        LOCAL_CACHE
    )


print(
    "\nTemporary model cache deleted."
)


print(
    "Done."
)